# Day 12

- **Streaming deduplication** — How to prevent duplicate transactions from being processed.
- **Checkpoint recovery** — How Spark resumes after a job stops or fails.
- **Exactly-once processing** — What guarantees Spark provides and where duplicates can still occur.

## Step 1 — Create a Delta source table- ransactions_dedup_source

In [0]:
# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField('transactionId', IntegerType(), False),
    StructField('custId', IntegerType(), False),
    StructField('amount', DoubleType(), False),
    StructField('serviceType', StringType(), False)
])

data = [
    (101, 1001, 100.0, "Wireless"),
    (102, 1002, 200.0, "Fiber"),
    (103, 1001, 150.0, "Internet"),
    (102, 1002, 200.0, "Fiber"),
    (104, 1003, 75.0, "Wireless")
]

df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_bronze.transactions_dedup_source")

## Step 2 — Read the source as a stream

In [0]:
dfStream = (
    spark.readStream
    .format("delta")
    .table(
        "workspace.customer360_bronze.transactions_dedup_source"
    )
)

## Step 3 — Apply streaming deduplication

Notice that we haven't used a watermark yet.

Spark must maintain the transaction IDs it has already seen across micro-batches. This is useful for learning, but retaining keys indefinitely can cause unbounded state growth in a long-running production pipeline.

In [0]:
dfDedup = (
    dfStream
    .dropDuplicates(["transactionId"])
)

## Step 4 — Write to a Delta table

NOTE: write the delta table to** SILVER level** as the deduplication is data quality task

In [0]:
checkpoint_path = ("/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/transactions_dedup_day12")

query = (
    dfDedup.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_silver.transactions_dedup_day12"
    )
)

query.awaitTermination()

## Step 5 — Verify the result

In [0]:
%sql
SELECT *
FROM workspace.customer360_silver.transactions_dedup_day12
ORDER BY transactionId;

## Step 6 — Append more transactions

append and run s2- s4.
** raw data will be in bronze.source table

In [0]:
new_data = [
    (103, 1001, 150.0, "Internet"),  # Duplicate
    (105, 1004, 125.0, "Fiber"),     # New
    (106, 1002, 80.0, "Wireless")    # New
]

dfNew = spark.createDataFrame(new_data, schema)

dfNew.write.format("delta").mode("append").saveAsTable(
    "workspace.customer360_bronze.transactions_dedup_source"
)

In [0]:
%sql
select * from workspace.customer360_bronze.transactions_dedup_source;

### NOTE-
- dropDuplicates(["transactionId"]) **still treats it as a duplicate.**  \
- It does not compare the amount or determine whether the newer record represents a correction.
- This is why **deduplication and CDC/upsert processing** are different problems.